# Risk-Stratified SHAP Networks for NHANES I dataset Random Forest Model

This notebook implements risk-stratified aggregation of SHAP values to identify different feature importance patterns across risk subgroups.

**Approach:** Stratify patients by predicted disease risk (low/high) and build separate networks for each group. This reveals how feature importance varies across risk levels.

**Mathematical Framework:**
- **Node weights** aggregate individual SHAP values across patients in each risk stratum
- **Edge weights** aggregate interaction values, with the denominator normalized using only the **lower triangular** interaction matrix (u > v), excluding diagonal elements

In [1]:
import os
import sys
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cgt_perezsechi.visualization.graph import draw
from cgt_perezsechi.manipulation.norm import normalize_psi, normalize_r

c:\Workspace\IJAR\IJAR-python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configure matplotlib for VS Code Jupyter
%matplotlib inline
# Set pandas display options to avoid truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_info_columns', 99999)

In [3]:
# Load data
X = pd.read_pickle("../../../../data/nhanesi/x_values.pkl")
num_patients = 500
X_shapley = X.iloc[:num_patients, :]
shap_values = np.load("../../../../data/nhanesi/rf/shap_values.npy")
shap_interaction_values = np.load("../../../../data/nhanesi/rf/shap_interaction_values.npy")

# Load y values (outcome/risk) from NHANES dataset
_, y = shap.datasets.nhanesi()
y_shapley = y[:num_patients]

## Stratify Patients by Actual Risk (y variable)

Use the actual outcome variable (y) to stratify patients into risk groups

In [4]:
from math import ceil

# Use actual outcome variable (y) for risk stratification
shap_values_clean = np.nan_to_num(shap_values)

# Stratify based on y values (actual disease outcome/risk)
# Sort and create tertiles based on y
sorted_indices = np.argsort(y_shapley)
n_low_risk = ceil(len(y_shapley) * 0.66)

low_risk_indices = sorted_indices[:n_low_risk]
high_risk_indices = sorted_indices[n_low_risk:]

# Create masks
low_risk_mask = np.isin(np.arange(len(y_shapley)), low_risk_indices)
high_risk_mask = np.isin(np.arange(len(y_shapley)), high_risk_indices)

# Display statistics
print(f"Risk stratification based on outcome variable (y):\n")
print(f"Low risk patients: {low_risk_mask.sum()}")
print(f"  y range: [{y_shapley[low_risk_indices].min():.4f}, {y_shapley[low_risk_indices].max():.4f}]")
print(f"  y mean: {y_shapley[low_risk_indices].mean():.4f}")

print(f"\nHigh risk patients: {high_risk_mask.sum()}")
print(f"  y range: [{y_shapley[high_risk_indices].min():.4f}, {y_shapley[high_risk_indices].max():.4f}]")
print(f"  y mean: {y_shapley[high_risk_indices].mean():.4f}")

Risk stratification based on outcome variable (y):

Low risk patients: 330
  y range: [-22.0833, -12.2500]
  y mean: -20.5359

High risk patients: 170
  y range: [-11.8333, 21.4167]
  y mean: 10.4657


## Function to Build Network for Risk Stratum

**Key Implementation Details:**

The function implements the following equations from the paper:

**Node weights (N*_i):**
$$\mathcal{N}^*_i = \frac{\sum_{t \in R_k} Sh_{i}(\Delta^{t})}{\sum_{u=1}^{n} \sum_{t \in R_k} |Sh_{u}(\Delta^{t})|}$$

**Edge weights (E*_ij):**
$$\mathcal{E}^*_{ij} = \frac{\sum_{t \in R_k} I_{ij}(\Delta^{t})}{\sum_{v=1}^{n} \sum_{u > v} \sum_{t \in R_k} |I_{uv}(\Delta^{t})|}$$

**Important:** The edge weight denominator sums only over the **lower triangular** part of the interaction matrix (where u > v), excluding diagonal elements. This differs from summing over all non-diagonal elements.

In [5]:
def build_network_for_stratum(patient_mask, shap_values, shap_interaction_values, X_shapley):
    """
    Build psi and r matrices for a specific patient stratum
    
    Implements equations:
    - Node weights: N*_i = sum_t(Sh_i) / sum_u sum_t |Sh_u|
    - Edge weights: E*_ij = sum_t(I_ij) / sum_v sum_{u>v} sum_t |I_uv|
      where u>v denotes the lower triangular matrix (excluding diagonal)
    """
    # Filter SHAP values for this stratum
    stratum_shap = shap_values[patient_mask]
    stratum_interaction = shap_interaction_values[patient_mask]
    
    # Calculate psi_1 (node weights)
    # Numerator: sum over patients for each feature
    # Denominator: sum of absolute values over all patients and all features
    sum_shap = np.sum(np.abs(stratum_shap), axis=(0, 1))
    psi_1 = pd.DataFrame()
    psi_1['value'] = np.sum(stratum_shap, axis=0) / sum_shap
    psi_1.set_index(X_shapley.columns, inplace=True)
    
    # Calculate r_1 (edge weights) - use only lower triangular matrix for denominator
    n_patients_stratum = stratum_interaction.shape[0]
    n_variables = stratum_interaction.shape[1]
    
    # Clean NaN values
    filtered_interaction = np.nan_to_num(stratum_interaction.copy())
    
    # Create lower triangular mask (u > v means row > col, excluding diagonal)
    lower_tri_mask = np.tril(np.ones((n_variables, n_variables), dtype=bool), k=-1)
    
    # Calculate denominator: sum of absolute values of lower triangular elements only
    # sum_v sum_{u>v} sum_t |I_uv(Delta^t)|
    sum_interaction_lower = 0.0
    for patient_idx in range(n_patients_stratum):
        sum_interaction_lower += np.sum(np.abs(filtered_interaction[patient_idx][lower_tri_mask]))
    
    # Calculate numerator: sum over patients for each (i,j) pair
    # sum_t I_ij(Delta^t)
    cumulative_interaction = np.sum(filtered_interaction, axis=0)
    
    # Zero out diagonal (remove self-interactions)
    np.fill_diagonal(cumulative_interaction, 0)
    
    # Calculate edge weights: E*_ij = numerator / denominator
    r_1 = pd.DataFrame(cumulative_interaction / sum_interaction_lower)
    r_1.rename(columns=dict(list(zip(r_1.columns, X_shapley.columns))), inplace=True)
    r_1.set_index(X_shapley.columns, inplace=True)
    
    # Normalize
    psi_2 = normalize_psi(psi_1)
    r_2 = normalize_r(r_1)
    
    return psi_1, r_1, psi_2, r_2

## Build Networks for Each Risk Stratum

### Low Risk Stratum

In [6]:
from IPython.display import display, Math

# Build networks for Low Risk Stratum
low_psi_1, low_r_1, low_psi_2, low_r_2 = build_network_for_stratum(
    low_risk_mask, shap_values_clean, shap_interaction_values, X_shapley
)

display("Low risk stratum")
display(Math(r"\mathcal{N}^*_i = \frac{\sum_{t \in R_{\mathrm{low}}} Sh_{i}(\Delta^{t})}{\sum_{u=1}^{n} \sum_{t \in R_{\mathrm{low}}} |Sh_{u}(\Delta^{t})|}"))
display(low_psi_2)
display(Math(r"\mathcal{E}^*_{ij} = \frac{\sum_{t \in R_{\mathrm{low}}} I_{ij}(\Delta^{t})}{\sum_{v=1}^{n} \sum_{u > v}^{n} \sum_{t \in R_{\mathrm{low}}} |I_{uv}(\Delta^{t})|}"))
display(low_r_2)

'Low risk stratum'

<IPython.core.display.Math object>

,value
sex_isFemale,-5.646734e-02
age,-1.000000e+00
physical_activity,-3.261239e-03
serum_albumin,-2.289790e-03
alkaline_phosphatase,-1.671969e-02
alkaline_phosphatase_isUnacceptable,3.728889e-05
alkaline_phosphatase_isTestnotdone,-5.004393e-06
SGOT,1.128833e-03
SGOT_isUnacceptable,1.098445e-03
SGOT_isTestnotdone,-5.995236e-06


<IPython.core.display.Math object>

,sex_isFemale,age,physical_activity,serum_albumin,alkaline_phosphatase,alkaline_phosphatase_isUnacceptable,alkaline_phosphatase_isTestnotdone,SGOT,SGOT_isUnacceptable,SGOT_isTestnotdone,BUN,BUN_isUnacceptable,BUN_isTestnotdone,calcium,calcium_isUnacceptable,calcium_isTestnotdone,creatinine,creatinine_isUnacceptable,creatinine_isTestnotdone,potassium,potassium_isUnacceptable,sodium,sodium_isUnacceptable,total_bilirubin,total_bilirubin_isUnacceptable,total_bilirubin_isTestnotdone,serum_protein,red_blood_cells,red_blood_cells_isUnacceptable,red_blood_cells_isBlankbutapplicable,white_blood_cells,white_blood_cells_isUnacceptable,white_blood_cells_isBlankbutapplicable,hemoglobin,hemoglobin_isMissing,hematocrit,hematocrit_isUnacceptable,hematocrit_isMissing,platelets_isNormal,platelets_isIncreased,platelets_isDecreased,platelets_isNoestimate,segmented_neutrophils,lymphocytes,monocytes,eosinophils,basophils,band_neutrophils,cholesterol,cholesterol_isMissing,urine_albumin_isNegative,urine_albumin_is>=30,urine_albumin_is>=100,urine_albumin_is>=300,urine_albumin_is>=1000,urine_albumin_isTrace,urine_albumin_isBlankbutapplicable,urine_glucose_isNegative,urine_glucose_isLight,urine_glucose_isMedium,urine_glucose_isDark,urine_glucose_isVerydark,urine_glucose_isTrace,urine_glucose_isBlankbutapplicable,urine_pH,urine_pH_isBlankbutapplicable,urine_hematest_isNegative,urine_hematest_isSmall,urine_hematest_isModerate,urine_hematest_isLarge,urine_hematest_isBlankbutapplicable,sedimentation_rate,sedimentation_rate_isBlankbutapplicable,uric_acid,uric_acid_isUnacceptable,uric_acid_isTestnotdone,systolic_blood_pressure,pulse_pressure,bmi
sex_isFemale,0.000000,0.989407,-4.443308e-02,-0.006557,-5.471343e-02,0.000000,-0.000142,-1.521633e-02,-0.001818,0.000000,-0.009290,-0.000261,0.0,-5.212470e-03,-0.000079,-5.304060e-04,-0.001987,0.000380,0.0,0.000125,-0.000144,0.003329,0.000000,-4.384479e-02,-0.000230,-0.001381,-0.014445,1.644108e-02,0.003147,-0.000502,1.021203e-02,9.804350e-05,-0.000706,-1.499807e-01,0.0,-0.114464,-0.000004,0.0,-0.000018,4.073759e-05,0.000000,0.0,0.001125,0.000360,-0.009840,-0.001580,-0.000312,4.939316e-04,1.213507e-03,0.0,6.618977e-04,0.000419,0.000000e+00,0.000000e+00,0.0,-6.493097e-04,0.000870,-5.870447e-03,0.000000e+00,0.000000,-1.049968e-04,-1.077314e-04,0.0,0.000244,-6.256193e-03,0.000536,0.000145,-7.534779e-05,0.0,0.0,-0.000167,-0.233862,1.279053e-04,-2.309803e-02,0.0,-0.000266,-1.644852e-01,-0.083867,-0.024958
age,0.989407,0.000000,1.191316e-01,0.075282,-1.847809e-01,-0.000947,-0.000348,-4.606253e-02,-0.004813,-0.000377,-0.048332,-0.000487,0.0,-1.678185e-02,-0.000261,-2.488937e-04,0.008229,-0.005801,0.0,-0.033650,-0.000210,0.009499,-0.000228,-4.352134e-02,-0.014033,-0.001779,0.041841,-8.903344e-02,-0.000005,-0.000078,-2.417281e-01,-1.269598e-03,0.009394,-3.716013e-01,0.0,-0.030079,-0.000150,0.0,-0.000694,-1.497035e-04,0.000008,0.0,-0.027444,-0.015750,-0.002011,-0.007120,-0.000906,-1.071246e-03,-2.694704e-01,0.0,-7.507261e-03,-0.003766,-4.229549e-03,-1.063255e-04,0.0,-1.776673e-03,-0.001669,-1.152148e-02,-1.859699e-04,-0.000854,-2.023196e-02,-2.998019e-04,0.0,0.001521,-3.098607e-02,0.000311,-0.001991,-2.824294e-04,0.0,0.0,0.000512,0.101471,-2.070580e-04,-4.905159e-02,0.0,0.000284,1.000000e+00,0.546046,-0.099580
physical_activity,-0.044433,0.119132,0.000000e+00,-0.002883,-8.588623e-04,0.000044,0.000000,-5.478403e-04,0.000000,0.000000,-0.000993,0.000000,0.0,-1.140744e-03,0.000000,3.311876e-07,0.000000,0.000048,0.0,-0.000651,0.000000,-0.000422,0.000000,-3.320107e-03,0.000028,0.000000,-0.001594,-3.098888e-03,0.000000,0.000000,-4.220895e-04,0.000000e+00,0.000050,-8.358735e-05,0.0,-0.001447,0.000000,0.0,0.000000,4.801235e-05,0.000000,0.0,0.000036,-0.000052,0.000072,-0.000504,0.000031,1.748099e-05,-1.161678e-05,0.0,-1.425762e-04,0.000000,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000041,-2.795860e-06,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,0.000000,-1.235148e-04,0.000000,0.000000,0.000000e+00,0.0,0.0,0.000000,-0.006005,0.

### High Risk Stratum

In [7]:
from IPython.display import display, Math

# Build networks for High Risk Stratum
high_psi_1, high_r_1, high_psi_2, high_r_2 = build_network_for_stratum(
    high_risk_mask, shap_values_clean, shap_interaction_values, X_shapley
)

display("High risk stratum")
display(Math(r"\mathcal{N}^*_i = \frac{\sum_{t \in R_{\mathrm{high}}} Sh_{i}(\Delta^{t})}{\sum_{u=1}^{n} \sum_{t \in R_{\mathrm{high}}} |Sh_{u}(\Delta^{t})|}"))
display(high_psi_2)
display(Math(r"\mathcal{E}^*_{ij} = \frac{\sum_{t \in R_{\mathrm{high}}} I_{ij}(\Delta^{t})}{\sum_{v=1}^{n} \sum_{u > v}^{n} \sum_{t \in R_{\mathrm{high}}} |I_{uv}(\Delta^{t})|}"))
display(high_r_2)

'High risk stratum'

<IPython.core.display.Math object>

,value
sex_isFemale,3.840140e-02
age,1.000000e+00
physical_activity,1.456978e-03
serum_albumin,4.679797e-03
alkaline_phosphatase,-2.788553e-03
alkaline_phosphatase_isUnacceptable,8.906540e-05
alkaline_phosphatase_isTestnotdone,1.412847e-05
SGOT,8.384209e-03
SGOT_isUnacceptable,1.560792e-03
SGOT_isTestnotdone,-4.467652e-06


<IPython.core.display.Math object>

,sex_isFemale,age,physical_activity,serum_albumin,alkaline_phosphatase,alkaline_phosphatase_isUnacceptable,alkaline_phosphatase_isTestnotdone,SGOT,SGOT_isUnacceptable,SGOT_isTestnotdone,BUN,BUN_isUnacceptable,BUN_isTestnotdone,calcium,calcium_isUnacceptable,calcium_isTestnotdone,creatinine,creatinine_isUnacceptable,creatinine_isTestnotdone,potassium,potassium_isUnacceptable,sodium,sodium_isUnacceptable,total_bilirubin,total_bilirubin_isUnacceptable,total_bilirubin_isTestnotdone,serum_protein,red_blood_cells,red_blood_cells_isUnacceptable,red_blood_cells_isBlankbutapplicable,white_blood_cells,white_blood_cells_isUnacceptable,white_blood_cells_isBlankbutapplicable,hemoglobin,hemoglobin_isMissing,hematocrit,hematocrit_isUnacceptable,hematocrit_isMissing,platelets_isNormal,platelets_isIncreased,platelets_isDecreased,platelets_isNoestimate,segmented_neutrophils,lymphocytes,monocytes,eosinophils,basophils,band_neutrophils,cholesterol,cholesterol_isMissing,urine_albumin_isNegative,urine_albumin_is>=30,urine_albumin_is>=100,urine_albumin_is>=300,urine_albumin_is>=1000,urine_albumin_isTrace,urine_albumin_isBlankbutapplicable,urine_glucose_isNegative,urine_glucose_isLight,urine_glucose_isMedium,urine_glucose_isDark,urine_glucose_isVerydark,urine_glucose_isTrace,urine_glucose_isBlankbutapplicable,urine_pH,urine_pH_isBlankbutapplicable,urine_hematest_isNegative,urine_hematest_isSmall,urine_hematest_isModerate,urine_hematest_isLarge,urine_hematest_isBlankbutapplicable,sedimentation_rate,sedimentation_rate_isBlankbutapplicable,uric_acid,uric_acid_isUnacceptable,uric_acid_isTestnotdone,systolic_blood_pressure,pulse_pressure,bmi
sex_isFemale,0.000000e+00,0.236012,1.577828e-02,8.712653e-03,8.936055e-03,0.000000,0.000025,2.837970e-03,-0.003340,0.000000,0.000894,4.890938e-05,0.0,-3.701279e-04,3.592109e-05,3.195256e-04,0.000082,-1.369970e-04,0.0,-1.380467e-05,0.000031,3.003114e-04,0.000000e+00,6.757278e-03,-0.000312,2.678461e-04,0.005688,-5.462662e-05,-0.000416,0.000067,1.897972e-02,-1.987248e-05,-0.000001,-2.232639e-02,0.0,-2.781350e-02,-0.000004,0.0,0.000017,7.075798e-07,0.000000,0.0,-6.060542e-04,1.616076e-03,0.001491,0.000410,-1.836529e-05,7.269977e-04,-3.708439e-03,0.0,-4.441574e-05,3.980831e-06,0.000000e+00,0.000000e+00,0.0,-1.669187e-04,1.401423e-04,0.000876,0.000000,0.000000,-1.331162e-06,2.924939e-05,0.0,-3.659453e-05,2.829677e-03,1.098844e-04,-1.800307e-04,1.471654e-05,0.0,0.0,0.000045,0.089120,-0.000008,-4.709536e-03,0.0,1.794677e-04,4.671038e-02,2.655753e-02,3.106324e-04
age,2.360118e-01,0.000000,2.218139e-02,-2.972304e-02,1.439130e-01,0.000654,0.000112,2.968523e-02,0.014573,-0.000008,0.012603,1.531124e-04,0.0,1.830122e-02,3.551387e-04,1.982234e-04,0.004431,2.018202e-03,0.0,2.577767e-02,0.000078,1.702577e-02,2.166931e-05,3.275816e-02,0.012049,5.479556e-04,0.055599,7.008232e-03,-0.000811,0.000410,5.912537e-02,4.321428e-04,-0.000522,1.344388e-01,0.0,2.240382e-02,0.000195,0.0,-0.000392,1.218241e-04,0.000188,0.0,-5.064682e-04,5.898011e-03,0.006885,0.004672,-6.975834e-04,-1.166932e-04,2.206004e-02,0.0,2.409127e-03,-3.599562e-03,1.075201e-03,6.997530e-05,0.0,-4.911436e-04,7.216995e-04,-0.005167,-0.000230,0.000848,-6.971132e-02,1.783624e-04,0.0,-1.902290e-05,-5.509199e-04,4.700140e-04,-2.403755e-05,9.692959e-04,0.0,0.0,0.000149,0.137805,-0.000158,6.116243e-03,0.0,4.681073e-05,-1.000000e+00,4.290670e-03,4.935310e-02
physical_activity,1.577828e-02,0.022181,0.000000e+00,7.840756e-04,8.539669e-05,-0.000059,0.000000,3.425602e-04,0.000000,0.000000,0.000091,0.000000e+00,0.0,4.324974e-05,0.000000e+00,5.582138e-08,0.000000,-9.897370e-06,0.0,-6.591991e-05,0.000000,-1.220769e-04,0.000000e+00,7.679285e-04,-0.000082,0.000000e+00,-0.000076,5.816842e-04,0.000000,0.000000,5.845996e-04,0.000000e+00,0.000010,-5.560097e-04,0.0,6.984610e-04,0.000000,0.0,0.000000,-7.066036e-06,0.000000,0.0,3.983553e-06,-4.212109e-05,-0.000110,0.000413,-1.746073e-04,2.920087e-06,2.088638e-04,0.0,-5.012650e-05,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,2.908359e-05,-0

## Visualization Setup

In [ ]:
shap_cmap = shap.plots.colors.red_blue
positive_color = shap_cmap(0.0)[:3]
negative_color = shap_cmap(1.0)[:3]

## Low Risk Network

In [ ]:
positive_alpha = 0.01
negative_alpha = 0.01
positive_beta = 0
negative_beta = 0

print("Low Risk Patients Network")
draw(
    psi=low_psi_2,
    r=low_r_2,
    positive_alpha=positive_alpha,
    negative_alpha=negative_alpha,
    positive_beta=positive_beta,
    negative_beta=negative_beta,
    negative_color=negative_color,
    positive_color=positive_color,
    output_path=os.path.join('..', '..', '..','..', 'result', 'nhanesi_rf_low_risk_stratus_network.jpg')
)

## High Risk Network

In [ ]:
positive_alpha = 0.025
negative_alpha = 0.025
positive_beta = 0
negative_beta = 0

print("High Risk Patients Network")
draw(
    psi=high_psi_2,
    r=high_r_2,
    positive_alpha=positive_alpha,
    negative_alpha=negative_alpha,
    positive_beta=positive_beta,
    negative_beta=negative_beta,
    negative_color=negative_color,
    positive_color=positive_color,
    output_path=os.path.join('..', '..', '..','..', 'result', 'nhanesi_rf_high_risk_stratus_network.jpg')
)